In [1]:
import pandas as pd
import time
from datetime import datetime

raw_df = pd.read_csv('raw_ingredient_batches.csv')
prep_df = pd.read_csv('prepped_items.csv')

raw_df['Received_Date'] = pd.to_datetime(raw_df['Received_Date'])
raw_df['Use_By_Date'] = pd.to_datetime(raw_df['Use_By_Date'])
prep_df['Prep_DateTime'] = pd.to_datetime(prep_df['Prep_DateTime'])
prep_df['Use_By_DateTime'] = pd.to_datetime(prep_df['Use_By_DateTime'])

TODAY = pd.Timestamp('2026-09-11')
print("Data loaded:", raw_df.shape, prep_df.shape)

Data loaded: (44, 11) (22, 12)


In [2]:
def days_to_use_by(use_by_date, today=None):
    if today is None:
        today = datetime.now()
    return (use_by_date - today).days

def usage_pace_status(row, today=None):
    if today is None:
        today = datetime.now()
    total_shelf_days = (row['Use_By_Date'] - row['Received_Date']).days
    days_elapsed = (today - row['Received_Date']).days
    days_remaining = (row['Use_By_Date'] - today).days
    if total_shelf_days <= 0 or row['Quantity_Received'] <= 0:
        return None
    pct_time_elapsed = days_elapsed / total_shelf_days
    pct_qty_used = row['Quantity_Used_So_Far'] / row['Quantity_Received']
    pace_ratio = (pct_qty_used / pct_time_elapsed) if pct_time_elapsed > 0 else None
    return {
        "days_remaining": days_remaining,
        "pct_time_elapsed": round(pct_time_elapsed * 100, 1),
        "pct_qty_used": round(pct_qty_used * 100, 1),
        "pace_ratio": round(pace_ratio, 2) if pace_ratio is not None else None,
    }
print("Tools 1 & 2 ready")

Tools 1 & 2 ready


In [3]:
def load_policy_sections(filepath):
    with open(filepath, 'r') as f:
        text = f.read()
    sections = {}
    parts = text.split('\n## ')
    for part in parts[1:]:
        lines = part.split('\n', 1)
        title = lines[0].strip()
        body = lines[1].strip() if len(lines) > 1 else ""
        sections[title] = body
    return sections

policy_sections = load_policy_sections('kitchen_waste_policy.md')

def get_policy_for_category(category, policy_sections):
    category_map = {
        "Seafood": "Section 7.1 — Seafood",
        "Meat & Poultry": "Section 7.2 — Meat & Poultry",
        "Dairy": "Section 7.3 — Dairy",
        "Produce": "Section 7.4 — Produce",
        "Dry Goods & Spices": "Section 7.5 — Dry Goods & Spices",
    }
    section_title = category_map.get(category)
    return policy_sections.get(section_title) if section_title else None

print("Policy sections loaded:", len(policy_sections))

Policy sections loaded: 11


In [4]:
def classify_raw_batch(row, policy_sections, today=None):
    if today is None:
        today = datetime.now()

    pace = usage_pace_status(row, today=today)
    if pace is None:
        return {"action": "ASK", "reason": "Invalid or missing data (zero quantity or shelf life)."}

    category = row['Category']
    days_remaining = pace['days_remaining']
    pace_ratio = pace['pace_ratio']
    days_elapsed = (today - row['Received_Date']).days

    if row['Quantity_Used_So_Far'] == 0 and days_elapsed >= 1:
        return {"action": "ASK", "reason": "No usage logged despite at least a day elapsed — no usage history to judge against."}

    if category != "Dry Goods & Spices":
        if pace_ratio is not None and pace_ratio > 3.5 and pace['pct_time_elapsed'] < 30:
            return {"action": "ASK", "reason": "Usage pace spiked sharply early in shelf life — unclear if this is a real trend or a one-off."}

    if days_remaining < 0:
        if category == "Dry Goods & Spices":
            return {"action": "FLAG_FOR_REVIEW", "reason": "Past use-by, but dry goods require human review before write-off (Section 7.5)."}
        return {"action": "WRITE_OFF", "reason": f"Past use-by date ({abs(days_remaining)} days expired)."}

    if category == "Seafood":
        if days_remaining <= 2:
            return {"action": "CONVERT_TO_PREP", "reason": "Seafood within 2-day at-risk window (Section 7.1) — usage pace not considered for this category."}

    elif category == "Meat & Poultry":
        if pace_ratio is not None and pace_ratio < 0.75:
            return {"action": "AT_RISK", "reason": f"Usage pace ({pace_ratio}) suggests fewer than 75% will be used in time (Section 7.2)."}

    elif category == "Dairy":
        total_shelf = (row['Use_By_Date'] - row['Received_Date']).days
        pct_shelf_remaining = (days_remaining / total_shelf * 100) if total_shelf > 0 else 0
        if pct_shelf_remaining <= 20:
            return {"action": "AT_RISK", "reason": f"Only {round(pct_shelf_remaining,1)}% of shelf life remaining (Section 7.3, urgent time-based threshold)."}
        if pace_ratio is not None and pace_ratio < 0.5:
            return {"action": "WATCH", "reason": f"Usage pace ({pace_ratio}) well behind elapsed time — early warning, shelf life still comfortable (Section 7.3, pace-based)."}

    elif category == "Produce":
        if pace_ratio is not None and pace_ratio < 0.70:
            return {"action": "AT_RISK", "reason": f"Usage pace ({pace_ratio}) suggests fewer than 70% will be used in time (Section 7.4)."}

    elif category == "Dry Goods & Spices":
        if days_remaining <= 7:
            return {"action": "AT_RISK", "reason": "Within 7 days of use-by (Section 7.5) — usage pace not a meaningful signal for this category."}

    return {"action": "ON_TRACK", "reason": "No risk threshold triggered."}

print("Tool 4 ready")

Tool 4 ready


In [5]:
def get_days_remaining(batch_id: str) -> dict:
    """Returns how many days remain until a raw ingredient batch's use-by date. Negative means already expired."""
    row = raw_df[raw_df['Batch_ID'] == batch_id]
    if len(row) == 0:
        return {"error": f"Batch {batch_id} not found."}
    row = row.iloc[0]
    days = days_to_use_by(row['Use_By_Date'], today=TODAY)
    return {"batch_id": batch_id, "ingredient": row['Ingredient_Name'], "category": row['Category'], "days_remaining": days}

def get_usage_pace(batch_id: str) -> dict:
    """Returns usage pace info for a raw ingredient batch: % of shelf life elapsed, % quantity used, and the pace ratio."""
    row = raw_df[raw_df['Batch_ID'] == batch_id]
    if len(row) == 0:
        return {"error": f"Batch {batch_id} not found."}
    row = row.iloc[0]
    pace = usage_pace_status(row, today=TODAY)
    if pace is None:
        return {"error": "Invalid data for this batch."}
    return {"batch_id": batch_id, **pace}

def get_policy_for_batch(batch_id: str) -> dict:
    """Retrieves the food-safety and waste-management policy for a raw ingredient batch,
    based on that batch's own category — automatically looked up, no need to specify category separately."""
    row = raw_df[raw_df['Batch_ID'] == batch_id]
    if len(row) == 0:
        return {"error": f"Batch {batch_id} not found."}
    row = row.iloc[0]
    category = row['Category']
    text = get_policy_for_category(category, policy_sections)
    if text is None:
        return {"error": f"No policy found for category '{category}'."}
    return {"batch_id": batch_id, "category": category, "policy_text": text}

# Quick local sanity check — no API call, free
print(get_days_remaining("RB-0027"))
print("Tool 5 ready")

{'batch_id': 'RB-0027', 'ingredient': 'Prawns', 'category': 'Seafood', 'days_remaining': 1}
Tool 5 ready


In [7]:
!pip install -q -U google-genai

from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')  # <-- paste the EXACT full secret name from your Secrets panel
client = genai.Client(api_key=api_key)

tools = [get_days_remaining, get_usage_pace, get_policy_for_batch]

print("Gemini client ready")

Gemini client ready


In [8]:
ACTION_DEFINITIONS = """
Action label definitions — use these precisely:
- ON_TRACK: No risk identified, no action needed now.
- WATCH: Early warning signal only (e.g. slow usage pace), not yet urgent — no immediate action required, just monitor.
- AT_RISK: Genuine risk of waste identified, needs attention, but no specific alternative action (like conversion) is recommended by policy — likely heading toward write-off if nothing changes.
- CONVERT_TO_PREP: Policy specifically recommends converting/using this item in prep or specials NOW as the corrective action (this is a more specific, actionable version of being at-risk — prefer this label over AT_RISK whenever the retrieved policy gives explicit convert-to-prep guidance).
- WRITE_OFF: Past use-by date, or policy dictates mandatory write-off.
- ASK: Data is ambiguous or insufficient to decide confidently.

IMPORTANT: If the retrieved policy explicitly recommends converting to prep or same-day use,
choose CONVERT_TO_PREP rather than the more generic AT_RISK.
"""

AGENT_SYSTEM_PROMPT = f"""You are a kitchen inventory assistant. You will be given a raw ingredient batch ID.

Investigate it using the tools available to you:
1. get_days_remaining — check how many days remain until use-by
2. get_usage_pace — check how quickly the batch is being used relative to time elapsed
3. get_policy_for_batch — retrieves the correct food-safety policy automatically for this batch

Call all three tools before deciding. Base your decision ONLY on what the tools actually returned.

{ACTION_DEFINITIONS}

Explain your reasoning step by step, quoting the specific policy rule that applies,
then state your final decision as "FINAL ACTION: <action>".
"""

MODEL_NAME = 'gemini-3.5-flash-lite'

def run_agent_on_batch(batch_id, model=MODEL_NAME, max_retries=4):
    prompt = f"{AGENT_SYSTEM_PROMPT}\n\nBatch ID to investigate: {batch_id}"
    for attempt in range(max_retries):
        try:
            fresh_chat = client.chats.create(
                model=model,
                config=types.GenerateContentConfig(tools=tools)
            )
            response = fresh_chat.send_message(prompt)
            return fresh_chat, response
        except Exception as e:
            error_str = str(e)
            if 'NOT_FOUND' in error_str or '404' in error_str:
                print("Model doesn't exist — not retrying.")
                raise
            if attempt < max_retries - 1:
                wait = 10 * (attempt + 1)
                print(f"Attempt {attempt+1} failed, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

print("Agent ready")

Agent ready


In [9]:
chat, response = run_agent_on_batch("RB-0027")
print(response.text)

Step-by-step investigation of batch **RB-0027** (Prawns, Category: Seafood):

1. **Days Remaining:** `get_days_remaining` shows there is **1 day remaining** until the use-by date.
2. **Usage Pace:** `get_usage_pace` shows that 50% of the shelf life has elapsed, but only 16.7% of the quantity has been used, giving a slow pace ratio of 0.33.
3. **Policy:** `get_policy_for_batch` provides the Seafood policy:
   - *At-risk threshold:* "Flag any seafood batch **2 days before** its use-by date, regardless of current usage pace." (With 1 day remaining, this batch is well past the 2-day threshold).
   - *Convert-to-prep guidance:* "Seafood nearing the at-risk window should be prioritized into same-day specials rather than held for later prep."
   - *Write-off rule:* "Any seafood batch past its use-by date must be written off immediately." (The batch still has 1 day remaining, so write-off is not yet required).

Since the batch has 1 day remaining and the policy explicitly states that seafood n

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import os
import shutil

project_folder = '/content/drive/MyDrive/expiry_waste_agent'
os.makedirs(project_folder, exist_ok=True)

for filename in ['raw_ingredient_batches.csv', 'prepped_items.csv', 'kitchen_waste_policy.md']:
    shutil.copy(filename, os.path.join(project_folder, filename))

print("Files saved to Drive:")
print(os.listdir(project_folder))

Files saved to Drive:
['raw_ingredient_batches.csv', 'prepped_items.csv', 'kitchen_waste_policy.md']


In [12]:
raw_df = pd.read_csv('/content/drive/MyDrive/expiry_waste_agent/raw_ingredient_batches.csv')

In [13]:
test_batches = {
    "already_expired": raw_df[raw_df['Scenario_Tag'] == 'already_expired'].iloc[0]['Batch_ID'],
    "ambiguous_no_usage_history": raw_df[raw_df['Scenario_Tag'] == 'ambiguous_no_usage_history'].iloc[0]['Batch_ID'],
    "dry_goods_normal": raw_df[raw_df['Scenario_Tag'] == 'dry_goods_normal'].iloc[0]['Batch_ID'],
}
print(test_batches)

{'already_expired': 'RB-0023', 'ambiguous_no_usage_history': 'RB-0013', 'dry_goods_normal': 'RB-0030'}


In [14]:
results_summary = []

for tag, batch_id in test_batches.items():
    print(f"\n{'='*60}")
    print(f"Testing: {tag} ({batch_id})")
    print('='*60)
    chat, response = run_agent_on_batch(batch_id)
    print(response.text)
    results_summary.append({"scenario": tag, "batch_id": batch_id, "response": response.text})
    time.sleep(15)  # space out calls to stay under the per-minute limit

print("\n\nAll tests complete.")


Testing: already_expired (RB-0023)
Based on the investigation and tools available:

1. **Days Remaining & Usage Pace:** The tool functions `get_days_remaining` and `get_usage_pace` encountered internal data type errors when queried for batch `RB-0023`.
2. **Policy Review:** The retrieved policy for this category (Meat & Poultry) states:
   > "- **Convert-to-prep guidance:** Poultry and meat nearing risk should be prioritized for prep that extends shelf life (e.g., marinating, cooking down into a sauce base) rather than held raw."
   > "- **At-risk threshold:** Flag when remaining shelf life is **30% or less** of total shelf life from receipt..."

Since the batch has been flagged as at-risk and the retrieved policy explicitly recommends converting poultry and meat nearing risk to prep rather than holding them raw, we follow the instruction: *"If the retrieved policy explicitly recommends converting to prep or same-day use, choose CONVERT_TO_PREP rather than the more generic AT_RISK."*


In [15]:
def usage_pace_status(row, today=None):
    if today is None:
        today = datetime.now()

    total_shelf_days = (row['Use_By_Date'] - row['Received_Date']).days
    days_elapsed = (today - row['Received_Date']).days
    days_remaining = (row['Use_By_Date'] - today).days

    if total_shelf_days <= 0 or row['Quantity_Received'] <= 0:
        return None

    pct_time_elapsed = days_elapsed / total_shelf_days
    pct_qty_used = float(row['Quantity_Used_So_Far']) / float(row['Quantity_Received'])
    pace_ratio = (pct_qty_used / pct_time_elapsed) if pct_time_elapsed > 0 else None

    return {
        "days_remaining": int(days_remaining),
        "pct_time_elapsed": float(round(pct_time_elapsed * 100, 1)),
        "pct_qty_used": float(round(pct_qty_used * 100, 1)),
        "pace_ratio": float(round(pace_ratio, 2)) if pace_ratio is not None else None,
    }

print("usage_pace_status fixed — now returns plain Python types")

usage_pace_status fixed — now returns plain Python types


In [16]:
time.sleep(20)
chat, response = run_agent_on_batch("RB-0023")
print(response.text)

Based on the tools and policy retrieved:

1. **Days Remaining**: Although the tool call encountered an internal type error, we can assess the batch status from the policy and usage triggers.
2. **Usage Pace / Policy**: The policy for Meat & Poultry states:
   > "- **At-risk threshold:** Flag when remaining shelf life is **30% or less** of total shelf life from receipt, OR usage pace suggests fewer than **75%** of units will be used by the use-by date — whichever comes first."
   > "- **Convert-to-prep guidance:** Poultry and meat nearing risk should be prioritized for prep that extends shelf life (e.g., marinating, cooking down into a sauce base) rather than held raw."

Because the policy explicitly dictates converting or prioritizing meat and poultry nearing risk into prep (such as marinating or cooking down into a sauce base) rather than holding them raw, the correct actionable label is `CONVERT_TO_PREP`.

FINAL ACTION: CONVERT_TO_PREP


In [18]:
result_days = get_days_remaining("RB-0023")
result_pace = get_usage_pace("RB-0023")

print("=== get_days_remaining ===")
for k, v in result_days.items():
    print(f"{k}: {v} (type: {type(v)})")

print("\n=== get_usage_pace ===")
for k, v in result_pace.items():
    print(f"{k}: {v} (type: {type(v)})")

TypeError: unsupported operand type(s) for -: 'str' and 'Timestamp'

In [19]:
print(raw_df.dtypes)

Batch_ID                 object
Ingredient_Name          object
Category                 object
Storage_Type             object
Received_Date            object
Use_By_Date              object
Quantity_Received       float64
Unit                     object
Unit_Cost               float64
Quantity_Used_So_Far    float64
Scenario_Tag             object
dtype: object


In [20]:
raw_df['Received_Date'] = pd.to_datetime(raw_df['Received_Date'])
raw_df['Use_By_Date'] = pd.to_datetime(raw_df['Use_By_Date'])
prep_df['Prep_DateTime'] = pd.to_datetime(prep_df['Prep_DateTime'])
prep_df['Use_By_DateTime'] = pd.to_datetime(prep_df['Use_By_DateTime'])
print(raw_df.dtypes)

Batch_ID                        object
Ingredient_Name                 object
Category                        object
Storage_Type                    object
Received_Date           datetime64[ns]
Use_By_Date             datetime64[ns]
Quantity_Received              float64
Unit                            object
Unit_Cost                      float64
Quantity_Used_So_Far           float64
Scenario_Tag                    object
dtype: object


In [21]:
result_days = get_days_remaining("RB-0023")
result_pace = get_usage_pace("RB-0023")
print(result_days)
print(result_pace)

{'batch_id': 'RB-0023', 'ingredient': 'Chicken (Bone-in)', 'category': 'Meat & Poultry', 'days_remaining': -2}
{'batch_id': 'RB-0023', 'days_remaining': -2, 'pct_time_elapsed': 166.7, 'pct_qty_used': 60.0, 'pace_ratio': 0.36}


In [22]:
chat, response = run_agent_on_batch("RB-0023")
print(response.text)

Step-by-step investigation of batch **RB-0023** (Chicken (Bone-in)):

1. **Days Remaining (`get_days_remaining`):** The batch has **-2 days remaining**, meaning it is already past its use-by date.
2. **Usage Pace (`get_usage_pace`):** The percentage of time elapsed is 166.7% and remaining days is -2.
3. **Policy (`get_policy_for_batch`):** The retrieved Meat & Poultry policy states: 
   > "- **Write-off rule:** Past use-by date, auto-write-off."

Since the batch is past its use-by date, the policy dictates mandatory write-off.

FINAL ACTION: WRITE_OFF


In [23]:
def get_prep_status(prep_id: str) -> dict:
    """Returns timing and usage status for a prepped item: hours remaining until use-by,
    usage pace, whether it's dairy-based, and its category (e.g. Bakery, Curry Base, Sauce)."""
    row = prep_df[prep_df['Prep_ID'] == prep_id]
    if len(row) == 0:
        return {"error": f"Prep {prep_id} not found."}
    row = row.iloc[0]

    total_shelf_hours = (row['Use_By_DateTime'] - row['Prep_DateTime']).total_seconds() / 3600
    hours_elapsed = (TODAY - row['Prep_DateTime']).total_seconds() / 3600
    hours_remaining = (row['Use_By_DateTime'] - TODAY).total_seconds() / 3600

    if total_shelf_hours <= 0 or row['Quantity_Prepped'] <= 0:
        return {"error": "Invalid data."}

    pct_time_elapsed = hours_elapsed / total_shelf_hours
    pct_qty_used = float(row['Quantity_Used_So_Far']) / float(row['Quantity_Prepped'])
    pace_ratio = (pct_qty_used / pct_time_elapsed) if pct_time_elapsed > 0 else None

    return {
        "prep_id": prep_id,
        "item_name": row['Item_Name'],
        "category": row['Category'],
        "is_dairy_based": bool(row['Is_Dairy_Based']),
        "hours_remaining": round(float(hours_remaining), 1),
        "pct_time_elapsed": round(float(pct_time_elapsed) * 100, 1),
        "pace_ratio": round(float(pace_ratio), 2) if pace_ratio is not None else None,
    }


def get_prep_source_check(prep_id: str) -> dict:
    """Checks whether a prepped item was made from raw ingredient batch(es) that were already
    at-risk or written-off AT THE TIME the item was prepped. Critical food-safety check."""
    row = prep_df[prep_df['Prep_ID'] == prep_id]
    if len(row) == 0:
        return {"error": f"Prep {prep_id} not found."}
    row = row.iloc[0]

    source_ids = str(row['Made_From_Batch_IDs']).split(';')
    findings = []
    for bid in source_ids:
        source_rows = raw_df[raw_df['Batch_ID'] == bid]
        if len(source_rows) > 0:
            source_row = source_rows.iloc[0]
            result = classify_raw_batch(source_row, policy_sections, today=row['Prep_DateTime'])
            findings.append({
                "batch_id": bid,
                "ingredient": source_row['Ingredient_Name'],
                "status_at_prep_time": result['action']
            })
    return {"prep_id": prep_id, "source_batches": findings}


def get_prep_policy(prep_id: str) -> dict:
    """Retrieves the relevant food-safety policy text for a prepped item, based on whether
    it's dairy-based, bakery, or a general prep — automatically determined from the item."""
    row = prep_df[prep_df['Prep_ID'] == prep_id]
    if len(row) == 0:
        return {"error": f"Prep {prep_id} not found."}
    row = row.iloc[0]

    if row['Is_Dairy_Based']:
        section = "Section 7.3 — Dairy"
    elif row['Category'] == 'Bakery':
        section = "Section 7.6a — Bakery & Baked Goods"
    else:
        section = "Section 7.6 — Prepped Items — General Rules"

    text = policy_sections.get(section)
    return {"prep_id": prep_id, "policy_section": section, "policy_text": text}


prep_tools = [get_prep_status, get_prep_source_check, get_prep_policy]
print("Prepped item tools ready")

Prepped item tools ready


In [24]:
PREP_AGENT_PROMPT = """You are a kitchen inventory assistant reviewing a PREPPED item (not raw ingredients).

Investigate using the tools:
1. get_prep_status — timing, usage pace, category, dairy status
2. get_prep_source_check — CRITICAL: checks if source raw ingredients were already at-risk/written-off when this was prepped
3. get_prep_policy — the relevant policy text for this item

Call all three tools before deciding.

OVERRIDE RULE: if get_prep_source_check shows ANY source batch had status "WRITE_OFF" at prep time,
you MUST choose URGENT_REVIEW regardless of the item's own timing status — this is a food-safety
override, not a normal risk assessment.

Otherwise choose from: ON_TRACK, AT_RISK, WATCH, WRITE_OFF, FLAG_FOR_REVIEW, ASK, URGENT_REVIEW

Explain your reasoning step by step, quoting relevant policy/tool findings, then state
"FINAL ACTION: <action>".
"""

def run_agent_on_prep(prep_id, model=MODEL_NAME, max_retries=4):
    prompt = f"{PREP_AGENT_PROMPT}\n\nPrep ID to investigate: {prep_id}"
    for attempt in range(max_retries):
        try:
            fresh_chat = client.chats.create(
                model=model,
                config=types.GenerateContentConfig(tools=prep_tools)
            )
            response = fresh_chat.send_message(prompt)
            return fresh_chat, response
        except Exception as e:
            error_str = str(e)
            if 'NOT_FOUND' in error_str or '404' in error_str:
                raise
            if attempt < max_retries - 1:
                wait = 10 * (attempt + 1)
                print(f"Attempt {attempt+1} failed, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

print("Prep agent ready")

Prep agent ready


In [25]:
print(get_prep_status("PP-0004"))
print(get_prep_source_check("PP-0004"))

{'prep_id': 'PP-0004', 'item_name': 'Tandoori Chicken Marinade', 'category': 'Marinated Protein', 'is_dairy_based': False, 'hours_remaining': 10.0, 'pct_time_elapsed': 79.2, 'pace_ratio': 0.51}
{'prep_id': 'PP-0004', 'source_batches': [{'batch_id': 'RB-0023', 'ingredient': 'Chicken (Bone-in)', 'status_at_prep_time': 'WRITE_OFF'}]}


In [26]:
chat, response = run_agent_on_prep("PP-0004")
print(response.text)

Step-by-step investigation for Prep ID **PP-0004** (Tandoori Chicken Marinade):

1. **Prep Status (`get_prep_status`)**: 
   - Item Name: Tandoori Chicken Marinade (Category: Marinated Protein)
   - Hours remaining: 10 hours
   - Percentage of time elapsed: 79.2%
   - Usage pace ratio: 0.51
   - Dairy-based: False

2. **Prep Source Check (`get_prep_source_check`)**:
   - Source batches: `RB-0023` (Chicken (Bone-in))
   - **Status at prep time: `WRITE_OFF`**

3. **Prep Policy (`get_prep_policy`)**:
   - Relevant policy: *Section 7.6 — Prepped Items — General Rules*
   - Highlights the traceability requirement: every prepped item must record which raw batch(es) it was made from to enable review of whether prep timing was reactive (made from stock that was already at risk) versus planned.

4. **Override Rule Evaluation**:
   - The override rule states: *"if get_prep_source_check shows ANY source batch had status 'WRITE_OFF' at prep time, you MUST choose URGENT_REVIEW regardless of the ite

In [29]:
raw_df['Agent_Action'] = raw_df.apply(lambda row: classify_raw_batch(row, policy_sections, today=TODAY)['action'], axis=1)

def classify_prep_with_override(prep_row, raw_df, policy_sections, today=None):
    result = classify_prepped_item(prep_row, raw_df, policy_sections, today=today)
    return result['action']

prep_df['Agent_Action'] = prep_df.apply(lambda row: classify_prep_with_override(row, raw_df, policy_sections, today=TODAY), axis=1)

print("=== RAW BATCH ACTIONS ===")
print(raw_df['Agent_Action'].value_counts())

print("\n=== PREPPED ITEM ACTIONS ===")
print(prep_df['Agent_Action'].value_counts())

# Calculate ₹ value at risk (AT_RISK + WRITE_OFF + URGENT_REVIEW batches)
at_risk_value = raw_df[raw_df['Agent_Action'].isin(['AT_RISK', 'WRITE_OFF'])].apply(
    lambda r: r['Unit_Cost'] * (r['Quantity_Received'] - r['Quantity_Used_So_Far']), axis=1
).sum()

print(f"\n💰 Estimated ₹ value in at-risk/write-off raw batches: ₹{at_risk_value:,.2f}")

=== RAW BATCH ACTIONS ===
Agent_Action
ON_TRACK           21
AT_RISK            13
WATCH               4
CONVERT_TO_PREP     3
ASK                 2
WRITE_OFF           1
Name: count, dtype: int64

=== PREPPED ITEM ACTIONS ===
Agent_Action
ON_TRACK         16
AT_RISK           5
URGENT_REVIEW     1
Name: count, dtype: int64

💰 Estimated ₹ value in at-risk/write-off raw batches: ₹19,925.54


In [28]:
def classify_prepped_item(prep_row, raw_df, policy_sections, today=None):
    if today is None:
        today = datetime.now()

    total_shelf_hours = (prep_row['Use_By_DateTime'] - prep_row['Prep_DateTime']).total_seconds() / 3600
    hours_elapsed = (today - prep_row['Prep_DateTime']).total_seconds() / 3600
    hours_remaining = (prep_row['Use_By_DateTime'] - today).total_seconds() / 3600

    if total_shelf_hours <= 0 or prep_row['Quantity_Prepped'] <= 0:
        return {"action": "ASK", "pattern_flag": None, "reason": "Invalid or missing data."}

    pct_time_elapsed = hours_elapsed / total_shelf_hours
    pct_qty_used = prep_row['Quantity_Used_So_Far'] / prep_row['Quantity_Prepped']
    pace_ratio = (pct_qty_used / pct_time_elapsed) if pct_time_elapsed > 0 else None

    pattern_flag = None
    source_was_written_off = False
    source_ids = str(prep_row['Made_From_Batch_IDs']).split(';')
    for bid in source_ids:
        source_rows = raw_df[raw_df['Batch_ID'] == bid]
        if len(source_rows) > 0:
            source_row = source_rows.iloc[0]
            source_result = classify_raw_batch(source_row, policy_sections, today=prep_row['Prep_DateTime'])
            if source_result['action'] in ('AT_RISK', 'CONVERT_TO_PREP', 'WRITE_OFF'):
                pattern_flag = f"Made from {bid} ({source_row['Ingredient_Name']}), which was already '{source_result['action']}' at prep time."
                if source_result['action'] == 'WRITE_OFF':
                    source_was_written_off = True
                break

    if hours_remaining < 0:
        if prep_row['Category'] == 'Bakery':
            action = "FLAG_FOR_REVIEW"
            reason = "Past use-by, but bakery items require human staleness check before write-off (Section 7.6a)."
        else:
            action = "WRITE_OFF"
            reason = f"Past use-by ({abs(round(hours_remaining,1))} hours expired)."
    elif prep_row['Is_Dairy_Based'] and hours_remaining <= 4:
        action = "AT_RISK"
        reason = "Dairy-based prep approaching its hard 24-hour limit (Section 7.3, no exceptions)."
    elif prep_row['Category'] == 'Bakery' and pace_ratio is not None and pace_ratio < 0.60:
        action = "AT_RISK"
        reason = f"Bakery item usage pace ({round(pace_ratio,2)}) below 60% threshold (Section 7.6a)."
    elif pace_ratio is not None and pace_ratio < 0.5 and pct_time_elapsed > 0.5:
        action = "AT_RISK"
        reason = f"Usage pace ({round(pace_ratio,2)}) well behind elapsed time, over halfway through shelf life."
    else:
        action = "ON_TRACK"
        reason = "No risk threshold triggered."

    if source_was_written_off:
        action = "URGENT_REVIEW"
        reason = "Source ingredient was already written off at prep time — food-safety override (Section 7.6)."

    return {"action": action, "pattern_flag": pattern_flag, "reason": reason}

print("classify_prepped_item ready")

classify_prepped_item ready
